# Deployment

Let's deploy the Foundation AI Foundation-Sec-8B-Reasoning model to **Amazon Bedrock** using Custom Model Import. <br>
You can use the model deployed by this notebook for inference. Refer to [the inference notebook](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/inference.ipynb) for sample code.

Bedrock imports model artifacts from S3 and creates a fully managed endpoint - no containers or instance types to configure.

As a prerequisite, you need:
- Model artifacts on S3 (BF16 safetensors format)
- IAM user with S3, Bedrock, and PassRole permissions
- A Bedrock service role with S3 read access

In [ ]:
%pip install -r ../requirements.txt --quiet

In [ ]:
import time
import json
import boto3

## Prerequisites (optional)

If your model is already on S3 and you already have a Bedrock service role, skip to **Configuration** below.

**Upload model to S3.** Bedrock imports from S3, not HuggingFace. Use the helper script to stream files directly (no local disk needed):

```bash
python ../upload_hf_to_s3.py --s3-uri s3://your-bucket/foundation-sec-8b-reasoning/
```

For gated models, set `HF_TOKEN` first. To upload a different model, pass `--repo-id`.

**Create Bedrock service role.** Bedrock needs an IAM role to read your S3 files. Uncomment and run the cell below to create one, or skip if you already have a role.

In [ ]:
# Uncomment this cell to create the Bedrock service role.
# Your IAM user needs iam:CreateRole and iam:AttachRolePolicy permissions.

# iam = boto3.client("iam")
# role_name = "bedrock-model-import-role"
#
# trust_policy = json.dumps({
#     "Version": "2012-10-17",
#     "Statement": [{
#         "Effect": "Allow",
#         "Principal": {"Service": "bedrock.amazonaws.com"},
#         "Action": "sts:AssumeRole",
#     }]
# })
#
# role = iam.create_role(
#     RoleName=role_name,
#     AssumeRolePolicyDocument=trust_policy,
#     Description="Allows Bedrock to read model artifacts from S3",
# )
#
# iam.attach_role_policy(
#     RoleName=role_name,
#     PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
# )
#
# role_arn = role["Role"]["Arn"]
# print(f"Created role: {role_arn}")
# print("Paste this ARN into BEDROCK_ROLE_ARN below.")

## Configuration

In [ ]:
S3_BUCKET_URI = ""  # e.g. "s3://your-bucket/foundation-sec-8b-reasoning/"
assert S3_BUCKET_URI, "Set S3_BUCKET_URI above before running"

BEDROCK_ROLE_ARN = ""  # e.g. "arn:aws:iam::123456789012:role/bedrock-model-import-role"
assert BEDROCK_ROLE_ARN, "Set BEDROCK_ROLE_ARN above before running"

AWS_REGION = "us-west-2"
IMPORTED_MODEL_NAME = "foundation-sec-8b-reasoning"

## Import Model

Submit the import job and wait for completion. This typically takes **5–15 minutes**.

You can also import models through the AWS Console: **Bedrock → Imported models → Import model**. See the [AWS docs](https://docs.aws.amazon.com/bedrock/latest/userguide/model-customization-import-model.html) for a walkthrough.

In [ ]:
bedrock_client = boto3.client("bedrock", region_name=AWS_REGION)

job_name = f"import-{IMPORTED_MODEL_NAME}-{int(time.time())}"

print(f"Model name : {IMPORTED_MODEL_NAME}")
print(f"S3 source  : {S3_BUCKET_URI}")
print(f"Region     : {AWS_REGION}")
print(f"Job name   : {job_name}")
print()

response = bedrock_client.create_model_import_job(
    jobName=job_name,
    importedModelName=IMPORTED_MODEL_NAME,
    roleArn=BEDROCK_ROLE_ARN,
    modelDataSource={
        "s3DataSource": {
            "s3Uri": S3_BUCKET_URI
        }
    }
)

job_arn = response["jobArn"]
print(f"Import job submitted: {job_arn}")

In [ ]:
print("Waiting for import to complete (typically 5-15 minutes)...\n")

while True:
    job_status = bedrock_client.get_model_import_job(jobIdentifier=job_arn)
    status = job_status["status"]

    if status == "Completed":
        model_arn = job_status["importedModelArn"]
        print(f"\nImport complete!")
        print(f"Model ARN: {model_arn}")
        print(f"\nShare this ARN with your team to use the model.")
        break
    elif status == "Failed":
        print(f"\nImport failed!")
        print(json.dumps(job_status, indent=2, default=str))
        raise RuntimeError("Bedrock import job failed")
    else:
        elapsed = int(time.time() - int(job_name.split("-")[-1]))
        print(f"  Status: {status} (elapsed: {elapsed // 60}m {elapsed % 60}s)", end="\r")
        time.sleep(15)

## Next Steps

The Model ARN is what you'll use for [inference](https://github.com/cisco-foundation-ai/cookbook/blob/main/3_adoptions/deployment/bedrock/foundation_sec_8b_reasoning/inference.ipynb). You can also view imported models in the AWS Console under **Bedrock → Imported models**.

## Cleanup

In [10]:
# Uncomment to delete the imported model and stop ongoing costs.
# Bedrock imported models bill continuously (~$1,080-1,440/month for 8B).
# S3 files remain intact (~$0.35/month) and you can re-import anytime.
#
# bedrock_client.delete_imported_model(modelIdentifier=model_arn)

In [ ]:
# Uncomment to delete the Bedrock service role created in the Prerequisites section.
#
# iam = boto3.client("iam")
# iam.detach_role_policy(
#     RoleName="bedrock-model-import-role",
#     PolicyArn="arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
# )
# iam.delete_role(RoleName="bedrock-model-import-role")
# print("Deleted role: bedrock-model-import-role")